In [28]:
!pip install fasttext

In [4]:
import pandas as pd
import re
import fasttext

In [5]:
df=pd.read_csv('Ecommerce_data.csv')
df.head()

,Text,label
0,Urban Ladder Eisner Low Back Study-Office Comp...,Household
1,"Contrast living Wooden Decorative Box,Painted ...",Household
2,IO Crest SY-PCI40010 PCI RAID Host Controller ...,Electronics
3,ISAKAA Baby Socks from Just Born to 8 Years- P...,Clothing & Accessories
4,Indira Designer Women's Art Mysore Silk Saree ...,Clothing & Accessories


In [6]:
df.label.replace('Clothing & Accessories','Clothing_Accessories',inplace=True)
df.label.unique()

/tmp/ipykernel_8808/438235586.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df.label.replace('Clothing & Accessories','Clothing_Accessories',inplace=True)


array(['Household', 'Electronics', 'Clothing_Accessories', 'Books'],
      dtype=object)

In [7]:
df['label']='__label__'+df['label'].astype(str)
df.head(3)

,Text,label
0,Urban Ladder Eisner Low Back Study-Office Comp...,__label__Household
1,"Contrast living Wooden Decorative Box,Painted ...",__label__Household
2,IO Crest SY-PCI40010 PCI RAID Host Controller ...,__label__Electronics


In [22]:
df['category_description']=df["label"]+" "+df["Text"]
df.head(5)

,Text,label,category_description
0,Urban Ladder Eisner Low Back Study-Office Comp...,__label__Household,__label__Household Urban Ladder Eisner Low Bac...
1,"Contrast living Wooden Decorative Box,Painted ...",__label__Household,__label__Household Contrast living Wooden Deco...
2,IO Crest SY-PCI40010 PCI RAID Host Controller ...,__label__Electronics,__label__Electronics IO Crest SY-PCI40010 PCI ...
3,ISAKAA Baby Socks from Just Born to 8 Years- P...,__label__Clothing_Accessories,__label__Clothing_Accessories ISAKAA Baby Sock...
4,Indira Designer Women's Art Mysore Silk Saree ...,__label__Clothing_Accessories,__label__Clothing_Accessories Indira Designer ...


In [9]:
text=df.category_description[0]
text

'__label__Household Urban Ladder Eisner Low Back Study-Office Computer Chair(Black) A study in simple. The Eisner study chair has a firm foam cushion, which makes long hours at your desk comfortable. The flexible meshed back is designed for air-circulation and support when you lean back. The curved arms provide ergonomic forearm support. Adjust the height using the gas lift to find that comfortable position and the nylon castors make it easy to move around your space. Chrome legs refer to the images for dimension details any assembly required will be done by the UL team at the time of delivery indoor use only.'

In [10]:
def preprocess(text):
  text=re.sub(r"[^\w\s\']"," ",text)
  text=re.sub(r" +"," ",text).strip()
  return text.lower()

preprocess(text)

'__label__household urban ladder eisner low back study office computer chair black a study in simple the eisner study chair has a firm foam cushion which makes long hours at your desk comfortable the flexible meshed back is designed for air circulation and support when you lean back the curved arms provide ergonomic forearm support adjust the height using the gas lift to find that comfortable position and the nylon castors make it easy to move around your space chrome legs refer to the images for dimension details any assembly required will be done by the ul team at the time of delivery indoor use only'

In [11]:
df['category_description']=df['category_description'].map(preprocess)
df.category_description[0]

'__label__household urban ladder eisner low back study office computer chair black a study in simple the eisner study chair has a firm foam cushion which makes long hours at your desk comfortable the flexible meshed back is designed for air circulation and support when you lean back the curved arms provide ergonomic forearm support adjust the height using the gas lift to find that comfortable position and the nylon castors make it easy to move around your space chrome legs refer to the images for dimension details any assembly required will be done by the ul team at the time of delivery indoor use only'

In [14]:
from sklearn.model_selection import train_test_split
train,test=train_test_split(df,test_size=0.2,random_state=42,stratify=df.label)
print(train.label.value_counts())
print(test.label.value_counts())

label
__label__Household               4800
__label__Electronics             4800
__label__Clothing_Accessories    4800
__label__Books                   4800
Name: count, dtype: int64
label
__label__Books                   1200
__label__Clothing_Accessories    1200
__label__Electronics             1200
__label__Household               1200
Name: count, dtype: int64


In [15]:
train.to_csv('ecommerce.train',columns=['category_description'],header=None,index=False)
test.to_csv('ecommerce.test',columns=['category_description'],header=None,index=False)

In [16]:
model=fasttext.train_supervised(input="ecommerce.train")
model.test("ecommerce.test")

(4798, 0.9701959149645686, 0.9701959149645686)

In [29]:
model.predict(preprocess("Indira Designer Women's Art Mysore Silk Saree" ) )

(('__label__clothing_accessories',), array([1.00000608]))